# SETU — best deployable model via SeqKD (resumable)

Trains the **SeqKD** student (the winning method: BLEU 17.1 vs 11.1 for
reference+DPO at 100k) at scale, quantises it to a ≤200 MB offline ONNX artifact,
tests it, and saves it to your Drive for download.

**Setup:** Runtime → Change runtime type → **GPU**, then **Runtime → Run all**
(Drive auth popup at cell 2).

**Resumable — just Run All again after a disconnect.** Everything expensive (data,
the distilled corpus, the trained checkpoint, the quantised model) is saved to
`MyDrive/setu_seqkd_deploy/` and marked **DONE** only on full success. Cell 2 prints
a status list; already-DONE steps are skipped in seconds. The one non-resumable
window is a training that dies mid-run (it restarts — but the ~30 min distill is
saved, so you never redo that). The teacher distill is greedy (`--beams 1`) for speed.

In [ ]:
import torch
print('cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - Runtime > Change runtime type > GPU')
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/setu_seqkd_deploy'   # persists across sessions
os.makedirs(WORK, exist_ok=True)
LIMIT = 200000                                      # SeqKD scales; 200k is a strong deployable target
print('persisting to', WORK, '\n')
for name, m in [('data','.done_data'), ('distill','.done_distill'),
                ('SeqKD train','.done_seqkd'), ('quantize','.done_quantize')]:
    print(f"  [{'DONE' if os.path.exists(f'{WORK}/{m}') else '  - '}] {name}")

In [ ]:
# clone + install + GPU configs; point data/ at Drive so the distilled corpus persists
%cd /content
!rm -rf /content/SETU_v2
!git clone https://github.com/GeekyRiolu/SETU_v2.git
%cd /content/SETU_v2/SETU
!pip -q install -e ".[data,teacher,quantize]"
!cp configs/model.gpu.yaml configs/model.yaml
!cp configs/training.gpu.yaml configs/training.yaml
!sed -i 's/device: cpu/device: cuda/' configs/teacher.yaml
import os
os.makedirs(f'{WORK}/data', exist_ok=True)
!rm -rf data && ln -s {WORK}/data data
print('data ->', os.path.realpath('data'))

In [ ]:
# 1) DATA (SeqKD needs no preference pairs) + 2) teacher-distilled corpus (greedy)
import os
if not os.path.exists(f'{WORK}/.done_data'):
    !setu-data --limit {LIMIT + 2000} && touch {WORK}/.done_data
else:
    print('data already saved - skipping')
if not os.path.exists(f'{WORK}/.done_distill'):
    !setu-distill --limit {LIMIT} --batch-size 32 --beams 1 && touch {WORK}/.done_distill
else:
    print('distilled corpus already saved - skipping')
_d = 'data/distilled/hin_Deva-eng_Latn/train.jsonl'
assert os.path.exists(_d) and os.path.getsize(_d) > 0, 'distill produced no corpus'
print('distilled rows:', sum(1 for _ in open(_d)))

In [ ]:
# 3) SeqKD training: SFT on teacher targets (no DPO). Eval on real references.
#    On success: report + full checkpoint saved to Drive so a resume skips it.
import os
if not os.path.exists(f'{WORK}/.done_seqkd'):
    !python scripts/train_full.py --train-corpus distilled --skip-dpo --limit {LIMIT} --dev-size 500 \
        && cp checkpoints/hin_Deva-eng_Latn/train_report.json {WORK}/report_seqkd.json \
        && rm -rf {WORK}/ckpt_seqkd && cp -r checkpoints/hin_Deva-eng_Latn {WORK}/ckpt_seqkd \
        && touch {WORK}/.done_seqkd \
        && echo '=== SeqKD trained + checkpoint saved to Drive ===' || echo '=== TRAINING FAILED - see above ==='
else:
    print('SeqKD already trained - restoring checkpoint from Drive')
    !mkdir -p checkpoints && rm -rf checkpoints/hin_Deva-eng_Latn && cp -r {WORK}/ckpt_seqkd checkpoints/hin_Deva-eng_Latn
import json
print(json.load(open(f'{WORK}/report_seqkd.json'))['sft_eval'])

In [ ]:
# 4) QUANTISE the SeqKD student -> INT8/INT4 ONNX, deploy under models/, score.
import os
if not os.path.exists(f'{WORK}/.done_quantize'):
    !setu-quantize --student sft \
        && rm -rf {WORK}/models_hin_en && cp -r models/hin_Deva-eng_Latn {WORK}/models_hin_en \
        && touch {WORK}/.done_quantize \
        && echo '=== quantised + saved to Drive ===' || echo '=== QUANTIZE FAILED - see above ==='
else:
    print('quantize already done - restoring from Drive')
    !mkdir -p models && rm -rf models/hin_Deva-eng_Latn && cp -r {WORK}/models_hin_en models/hin_Deva-eng_Latn
!setu-report --offline-proof

In [ ]:
# 5) TEST the deployed model offline
import sys; sys.path.insert(0, 'src')
from setu.inference.engine import InferenceEngine
eng = InferenceEngine(models_root='models')
print('using trained model:', not eng.is_stub)
for s in ['भारत एक विशाल देश है।', 'मुझे किताबें पढ़ना पसंद है।', 'आज मौसम अच्छा है।',
          'यह एक अच्छी किताब है।']:
    print(f'{s}  ->  {eng.translate(s, "hi", "en").translated_text}')

In [ ]:
# 6) SAVE the deployable model (zip) to Drive for download
!cd models && zip -qr {WORK}/setu_seqkd_model.zip hin_Deva-eng_Latn
print('deployable model saved to:', f'{WORK}/setu_seqkd_model.zip')
print('Download from Drive, then locally: unzip into SETU/models/ and run')
print('  python setu_cli.py --src hi --tgt en --text "भारत एक विशाल देश है।"')